# Market Data Cleaning & Standardisation Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

Cleans and standardises the OHLCV market data CSVs (one per asset) collected from CryptoCompare, preparing them for merging with the sentiment data in the subsequent analysis stage. Each file is processed independently and saved separately to preserve the per-asset structure.

**Workflow:**
1. Upload all per-asset market data CSVs at once.
2. Run cleaning and standardisation on each file.
3. Download all cleaned files as a ZIP.

**Processing steps applied to each file:**
- Standardise date format to YYYY-MM-DD.
- Enforce the dissertation study period (2020 to 2024).
- Validate OHLCV integrity so that high >= low, prices are positive, and volume is non-negative.
- Fill any missing days by forward-filling prices and setting volume to zero (crypto trades 24/7 so these gaps should be rare).
- Compute derived features needed for sentiment-price analysis such as returns, volatility, intraday range, and volume change.
- Detect and optionally winsorise extreme outliers in returns.
- Drop duplicate dates.

## 0. Setup & Dependencies

In [ ]:
!pip install -q pandas numpy

import pandas as pd
import numpy as np
import re
import os
import shutil
from datetime import datetime
from pathlib import Path
from google.colab import files

os.makedirs('data/raw/market', exist_ok=True)
os.makedirs('data/processed/market', exist_ok=True)
os.makedirs('data/processed/market/stats', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Configuration

In [ ]:
# --- Study period ---
START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2024-12-31')

# --- Asset metadata (matched to filenames) ---
ASSET_METADATA = {
    'bitcoin':   {'symbol': 'BTC',  'category': 'established'},
    'ethereum':  {'symbol': 'ETH',  'category': 'established'},
    'dogecoin':  {'symbol': 'DOGE', 'category': 'meme_coin'},
    'shiba_inu': {'symbol': 'SHIB', 'category': 'meme_coin'},
    'tether':    {'symbol': 'USDT', 'category': 'stablecoin_control'},
}

# --- Outlier handling ---
WINSORISE_LOWER = 0.001  # 0.1st percentile
WINSORISE_UPPER = 0.999  # 99.9th percentile
APPLY_WINSORISATION = True

print(f'Study period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Assets: {list(ASSET_METADATA.keys())}')
print(f'Winsorisation: {APPLY_WINSORISATION}')

Study period: 2020-01-01 to 2024-12-31
Assets: ['bitcoin', 'ethereum', 'dogecoin', 'shiba_inu', 'tether']
Winsorisation: True


In [ ]:
# --- Helper functions ---

def detect_asset_from_filename(filename):
    fname_lower = filename.lower()
    for asset in ['shiba_inu', 'shibainu', 'shib',
                  'bitcoin', 'ethereum', 'dogecoin', 'tether']:
        if asset in fname_lower:
            if asset in ('shibainu', 'shib'):
                return 'shiba_inu'
            return asset
    symbol_map = {'btc': 'bitcoin', 'eth': 'ethereum',
                  'doge': 'dogecoin', 'usdt': 'tether'}
    for sym, asset in symbol_map.items():
        if re.search(r'\b' + sym + r'\b', fname_lower):
            return asset
    return None

def detect_columns(df):
    cols = {col.lower().strip(): col for col in df.columns}
    mapping = {}
    for candidate in ['date', 'timestamp', 'datetime', 'day']:
        if candidate in cols:
            mapping['date'] = cols[candidate]; break
    for std, candidates in {
        'open':  ['open', 'opening_price', 'price_open'],
        'high':  ['high', 'day_high', 'price_high'],
        'low':   ['low', 'day_low', 'price_low'],
        'close': ['close', 'closing_price', 'price_close', 'price_usd'],
    }.items():
        for c in candidates:
            if c in cols:
                mapping[std] = cols[c]; break
    for std, candidates in {
        'volume_usd':  ['volume_usd', 'volumeto', 'quote_volume'],
        'volume_base': ['volumefrom', 'vol_base', 'volume_base', 'base_volume'],
    }.items():
        for c in candidates:
            if c in cols:
                mapping[std] = cols[c]; break
    return mapping

def winsorise(series, lower=0.001, upper=0.999):
    if series.empty or series.isna().all():
        return series
    return series.clip(lower=series.quantile(lower),
                       upper=series.quantile(upper))

print('Helper functions loaded.')

Helper functions loaded.


## 2. Upload All Market Data CSVs

Select all your per-asset market data CSVs at once.

In [ ]:
print('Upload all your market data CSVs (one per asset):\n')
uploaded = files.upload()
for fname, content in uploaded.items():
    dest = f'data/raw/market/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content)/1e6:.2f} MB)')
print(f'\n{len(uploaded)} file(s) uploaded.')

Upload all your market data CSVs (one per asset):



Saving all_assets_daily.csv to all_assets_daily.csv
Saving bitcoin_daily.csv to bitcoin_daily.csv
Saving dogecoin_daily.csv to dogecoin_daily.csv
Saving ethereum_daily.csv to ethereum_daily.csv
Saving shiba_inu_daily.csv to shiba_inu_daily.csv
Saving tether_daily.csv to tether_daily.csv
  Saved: data/raw/market/all_assets_daily.csv (1.84 MB)
  Saved: data/raw/market/bitcoin_daily.csv (0.40 MB)
  Saved: data/raw/market/dogecoin_daily.csv (0.39 MB)
  Saved: data/raw/market/ethereum_daily.csv (0.39 MB)
  Saved: data/raw/market/shiba_inu_daily.csv (0.29 MB)
  Saved: data/raw/market/tether_daily.csv (0.36 MB)

6 file(s) uploaded.


## 3. Cleaning & Standardisation Function

Cleans the OHLCV data, validates integrity, fills any gaps since crypto markets trade 24/7, and computes the derived features needed for sentiment-price analysis.

In [ ]:
def clean_market_file(input_path, output_path, stats_path, asset_name):
    print('\n' + '=' * 60)
    print(f'CLEANING: {os.path.basename(input_path)}')
    print('=' * 60)

    try:
        df = pd.read_csv(input_path, on_bad_lines='skip',
                         engine='python', encoding='utf-8',
                         encoding_errors='ignore')
    except Exception as e:
        print(f'  ERROR loading file: {e}')
        return None

    initial = len(df)
    print(f'  Loaded: {initial:,} rows')
    print(f'  Original columns: {list(df.columns)}')

    col_map = detect_columns(df)
    print(f'  Detected mapping: {col_map}')

    required = ['date', 'close']
    missing = [c for c in required if c not in col_map]
    if missing:
        print(f'  ERROR: Missing required columns: {missing}')
        return None

    df = df.rename(columns={v: k for k, v in col_map.items()})

    stats = {k: 0 for k in ['initial', 'dropped_invalid_date',
                             'dropped_date_range', 'dropped_invalid_ohlcv',
                             'dropped_duplicate', 'gaps_filled',
                             'winsorised_returns', 'kept']}
    stats['initial'] = initial

    # 1. Parse and validate dates.
    df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
    if df['date'].dt.tz is not None:
        df['date'] = df['date'].dt.tz_localize(None)
    b = len(df); df = df[df['date'].notna()]
    stats['dropped_invalid_date'] = b - len(df)
    df['date'] = df['date'].dt.normalize()

    # 2. Filter to study period.
    b = len(df)
    df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)]
    stats['dropped_date_range'] = b - len(df)

    # 3. Convert OHLCV columns to numeric.
    for col in ['open', 'high', 'low', 'close', 'volume_usd', 'volume_base']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    b = len(df)
    df = df[df['close'].notna() & (df['close'] > 0)]
    if all(c in df.columns for c in ['open', 'high', 'low']):
        df = df[
            (df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0) &
            (df['high'] >= df['low']) &
            (df['high'] >= df['open']) & (df['high'] >= df['close']) &
            (df['low'] <= df['open']) & (df['low'] <= df['close'])
        ]
    stats['dropped_invalid_ohlcv'] = b - len(df)

    # 4. Deduplicate by date.
    b = len(df)
    df = df.sort_values('date').drop_duplicates(subset=['date'], keep='last')
    stats['dropped_duplicate'] = b - len(df)

    # 5. Reindex to complete daily range.
    if len(df) == 0:
        print('  ERROR: No valid rows remain after filtering.')
        return None
    full_range = pd.date_range(
        start=max(df['date'].min(), START_DATE),
        end=min(df['date'].max(), END_DATE),
        freq='D')
    df = df.set_index('date').reindex(full_range)
    gaps_before = df['close'].isna().sum()
    price_cols = [c for c in ['open', 'high', 'low', 'close'] if c in df.columns]
    df[price_cols] = df[price_cols].ffill()
    vol_cols = [c for c in ['volume_usd', 'volume_base'] if c in df.columns]
    df[vol_cols] = df[vol_cols].fillna(0)
    stats['gaps_filled'] = int(gaps_before)
    df = df.reset_index().rename(columns={'index': 'date'})

    # 6. Attach asset metadata.
    meta = ASSET_METADATA.get(asset_name, {'symbol': asset_name.upper(),
                                            'category': 'unknown'})
    df['asset'] = asset_name
    df['symbol'] = meta['symbol']
    df['category'] = meta['category']

    # 7. Compute derived features.
    df['price_usd'] = df['close']
    df['daily_return_pct'] = df['close'].pct_change() * 100
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df['volatility_7d'] = df['log_return'].rolling(window=7, min_periods=3).std()
    df['volatility_30d'] = df['log_return'].rolling(window=30, min_periods=10).std()
    if all(c in df.columns for c in ['high', 'low', 'open']):
        df['intraday_range_pct'] = ((df['high'] - df['low']) /
                                     df['open'].replace(0, np.nan)) * 100
    if 'volume_usd' in df.columns:
        df['volume_change_pct'] = df['volume_usd'].pct_change() * 100
        df['volume_ma_7d'] = df['volume_usd'].rolling(window=7,
                                                       min_periods=3).mean()

    # 8. Optional winsorisation.
    if APPLY_WINSORISATION and 'log_return' in df.columns:
        original = df['log_return'].copy()
        df['log_return'] = winsorise(df['log_return'],
                                      WINSORISE_LOWER, WINSORISE_UPPER)
        df['daily_return_pct'] = winsorise(df['daily_return_pct'],
                                            WINSORISE_LOWER, WINSORISE_UPPER)
        stats['winsorised_returns'] = int(
            (original != df['log_return']).sum())

    df['date'] = df['date'].dt.strftime('%Y-%m-%d')

    ordered_cols = (
        ['date', 'asset', 'symbol', 'category'] +
        [c for c in ['open','high','low','close','price_usd',
                     'volume_usd','volume_base'] if c in df.columns] +
        [c for c in ['daily_return_pct','log_return',
                     'volatility_7d','volatility_30d',
                     'intraday_range_pct','volume_change_pct',
                     'volume_ma_7d'] if c in df.columns]
    )
    df = df[ordered_cols]

    stats['kept'] = len(df)
    df.to_csv(output_path, index=False)

    print(f'\n  --- Summary ---')
    for key, val in stats.items():
        print(f'  {key:28s} {val:>8,}')
    print(f'  Date range: {df["date"].min()} to {df["date"].max()}')
    print(f'  Mean price: ${df["price_usd"].mean():,.6f}')
    print(f'  Saved: {output_path}')

    if stats_path:
        with open(stats_path, 'w') as f:
            f.write(f'Market Data Cleaning Statistics\nFile: {input_path}\n')
            f.write(f'Run: {datetime.now().isoformat()}\n' + '=' * 50 + '\n\n')
            for key, val in stats.items():
                f.write(f'{key}: {val:,}\n')

    return stats

print('Cleaning function loaded.')

Cleaning function loaded.


## 4. Process All Uploaded Files

In [ ]:
INPUT_DIR = 'data/raw/market'
OUTPUT_DIR = 'data/processed/market'
STATS_DIR = 'data/processed/market/stats'

csv_files = sorted([f for f in os.listdir(INPUT_DIR)
                    if f.endswith('.csv') and not f.startswith('all_')])

if not csv_files:
    print(f'No CSV files found in {INPUT_DIR}. Upload files in Step 2 first.')
else:
    print('=' * 60)
    print(f'PROCESSING {len(csv_files)} MARKET DATA FILE(S)')
    print('=' * 60)
    for f in csv_files:
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1e6
        print(f'  {f} ({size_mb:.2f} MB)')

    overall_stats = {}
    for fname in csv_files:
        input_path = os.path.join(INPUT_DIR, fname)
        output_path = os.path.join(OUTPUT_DIR,
                                    fname.replace('.csv', '_cleaned.csv'))
        stats_path = os.path.join(STATS_DIR,
                                   fname.replace('.csv', '_stats.txt'))
        asset = detect_asset_from_filename(fname)
        if not asset:
            print(f'\n  Warning: could not infer asset for {fname}. Skipping.')
            continue
        print(f'\n  Detected asset: {asset} '
              f'({ASSET_METADATA[asset]["symbol"]}, '
              f'{ASSET_METADATA[asset]["category"]})')
        stats = clean_market_file(input_path, output_path, stats_path, asset)
        if stats:
            overall_stats[fname] = stats

    print('\n' + '=' * 60)
    print('OVERALL SUMMARY')
    print('=' * 60)
    total_kept = sum(s['kept'] for s in overall_stats.values())
    print(f'Files processed:   {len(overall_stats)}')
    print(f'Total kept rows:   {total_kept:,}')
    print(f'\nPer-file results:')
    for fname, stats in overall_stats.items():
        print(f'  {fname:40s} {stats["kept"]:>6,} days kept')

PROCESSING 5 MARKET DATA FILE(S)
  bitcoin_daily.csv (0.40 MB)
  dogecoin_daily.csv (0.39 MB)
  ethereum_daily.csv (0.39 MB)
  shiba_inu_daily.csv (0.29 MB)
  tether_daily.csv (0.36 MB)

  Detected asset: bitcoin (BTC, established)

CLEANING: bitcoin_daily.csv
  Loaded: 1,827 rows
  Original columns: ['date', 'asset', 'symbol', 'category', 'open', 'high', 'low', 'close', 'price_usd', 'volume_usd', 'volumefrom', 'daily_range_pct', 'daily_return_pct', 'log_return', 'volatility_7d', 'volatility_30d', 'volume_change_pct']
  Detected mapping: {'date': 'date', 'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close', 'volume_usd': 'volume_usd', 'volume_base': 'volumefrom'}

  --- Summary ---
  initial                         1,827
  dropped_invalid_date                0
  dropped_date_range                  0
  dropped_invalid_ohlcv               0
  dropped_duplicate                   0
  gaps_filled                         0
  winsorised_returns                  5
  kept             

## 5. Download All Cleaned Files

In [ ]:
zip_name = 'market_data_cleaned'
shutil.make_archive(zip_name, 'zip', 'data/processed', 'market')
zip_path = f'{zip_name}.zip'
print(f'Archive: {zip_path} '
      f'({os.path.getsize(zip_path) / 1e6:.2f} MB)')
files.download(zip_path)
print('Download started.')

Archive: market_data_cleaned.zip (0.77 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
